# Notebook PyTorch Custom CNN per regressione numero tazze

In [ ]:
# Install missing
!pip install pandas pillow torch torchvision matplotlib tqdm

In [ ]:
# ======================================
# Import librerie e setup
# ======================================
import os
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

# Impostazioni Jupyter
%matplotlib inline
plt.style.use('seaborn-v0_8')


# Controllo device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device usato:", device)
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ======================================
# Caricamento CSV multipli e merge e distribuzione
# ======================================
import glob
DATA_DIR = "./DatasetTazzePerRegressione"

# Trova tutti i file objects_count.csv nelle sottocartelle
csv_files = glob.glob(os.path.join(DATA_DIR, "**", "objects_count.csv"), recursive=True)

print("CSV trovati:")
for f in csv_files:
    print(f)

# Carica e concatena tutti i CSV
dfs = [pd.read_csv(f) for f in csv_files]
dataset_df = pd.concat(dfs, ignore_index=True)

print("\nTotale immagini:", len(dataset_df))
dataset_df.head()



In [ ]:

# Distribuzione ObjectsCount
distribution = dataset_df["ObjectsCount"].value_counts().sort_index()
print("Distribuzione ObjectsCount:")
print(distribution)

In [ ]:
class CupsDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        batch = str(self.df.loc[idx, "Batch"])          # FORZA stringa
        image_name = str(self.df.loc[idx, "Image"])     # FORZA stringa
        label = torch.tensor(self.df.iloc[idx]["ObjectsCount"], dtype=torch.float32)


        img_path = os.path.join(
            self.root_dir,
            batch,
            "non_etichettate",
            f"{batch}_{image_name}"
        )

        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Immagine non trovata: {img_path}")

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
#obiettivo calcolare media e dev stand
from torch.utils.data import DataLoader
from torchvision import transforms

stats_transform = transforms.Compose([
    transforms.Resize((500, 500)),  # stessa dimensione usata per training
    transforms.ToTensor(),           
])

stats_dataset = CupsDataset(
    dataset_df,
    root_dir=DATA_DIR,
    transform=stats_transform
)

stats_loader = DataLoader(
    stats_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

mean = torch.zeros(3)
std = torch.zeros(3)
total_images = 0

for images, _ in stats_loader:
    batch_size = images.size(0)
    images = images.view(batch_size, 3, -1)

    mean += images.mean(dim=2).sum(dim=0)
    std += images.std(dim=2).sum(dim=0)
    total_images += batch_size

mean /= total_images
std /= total_images

print("Mean:", mean)
print("Std:", std)


In [ ]:
#da usare in training
transform = transforms.Compose([
    transforms.Resize((500, 500)),
    transforms.RandomRotation(5),
    transforms.RandomHorizontalFlip(0.3),
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.3,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Lambda(
        lambda x: torch.clamp(x + 0.01 * torch.randn_like(x), 0.0, 1.0)
    ),
    transforms.Normalize(
        mean=mean.tolist(),
        std=std.tolist()
    )
])

#da usare in inferenza
inference_transform = transforms.Compose([
    transforms.Resize((500, 500)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean.tolist(),
        std=std.tolist()
    )
])



In [ ]:
# ======================================
# Dataset BASE (senza transform)
# ======================================
full_dataset = CupsDataset(
    dataset_df,
    root_dir=DATA_DIR,
    transform=None
)

# ======================================
# Split train / validation
# ======================================
total_len = len(full_dataset)
train_len = int(0.8 * total_len)
val_len = total_len - train_len

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_len, val_len]
)

# ======================================
# Applica le transform DOPO lo split
# ======================================
train_dataset.dataset.transform = transform    #train_transform
val_dataset.dataset.transform = inference_transform

# ======================================
# DataLoader
# ======================================
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)


In [ ]:
print(total_len)
print(train_len)
print(val_len)

In [ ]:
images, labels = next(iter(train_loader))
#controllo non ci siano valori invalidi (problema NaN) 
print("Images NaN:", torch.isnan(images).any().item())
print("Images Inf:", torch.isinf(images).any().item())
print("Min:", images.min().item())
print("Max:", images.max().item())
print("Shape images:", images.shape)

print("Label prima immagine:", labels[0])


#Output atteso:
#Images NaN: False
#Images Inf: False
#Min: circa -2 / -1
#Max: circa 2 / 3

In [ ]:
# ======================================
# Modello Custom CNN : Regressore (Fix automatico flatten per 500x500)
# ======================================
import torch
from torch import nn

class CustomCNN(nn.Module):
    def __init__(self, input_size=(3, 500, 500)):
        super(CustomCNN, self).__init__()

        # ----------------------------
        # Feature extractor
        # ----------------------------
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # /2 -> 500 -> 250
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # /2 -> 250 -> 125
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # /2 -> 125 -> 62
        )

        # ----------------------------
        # Calcolo automatico dimensione flatten
        # ----------------------------
        with torch.no_grad():
            dummy = torch.zeros(1, *input_size)  # batch 1, canali x H x W
            dummy = self.features(dummy)
            self.flatten_dim = dummy.numel()  # totale elementi dopo conv

        # ----------------------------
        # Classifier
        # ----------------------------
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)  # output regressione
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ----------------------------
# Spostamento su GPU (se disponibile)
# ----------------------------
model = CustomCNN().to(device)
print("Modello pronto. Flatten dim:", model.flatten_dim)


In [ ]:
# ======================================
# Loss MAE e ottimizzatore 
# ======================================
criterion = nn.L1Loss() 
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
print("Num batch train_loader:", len(train_loader))
print("Num batch val_loader:", len(val_loader))


In [ ]:
# ======================================
# Riprendi training da checkpoint se esiste
# ======================================
import glob

# Cartella dove salvi i checkpoint
checkpoint_dir = "./checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# Cerca tutti i checkpoint disponibili
checkpoint_files = glob.glob(os.path.join(checkpoint_dir, "checkpoint_epoch_*.pt"))

start_epoch = 0
train_losses = []
val_losses = []

if checkpoint_files:
    # Prendi l'ultimo checkpoint disponibile
    latest_checkpoint = max(checkpoint_files, key=os.path.getctime)
    print(f"Trovato checkpoint, riprendo da: {latest_checkpoint}")
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] +1
    train_losses = checkpoint.get('train_losses', [])
    val_losses = checkpoint.get('val_losses', [])
    print(f"Parto dall'epoca {start_epoch}")
else:
    print(" Nessun checkpoint trovato, parto da zero")


In [ ]:
#per loggare training data
import os

log_file = "training_log.csv"

# scrive l'header solo se il file non esiste
if not os.path.exists(log_file):
    with open(log_file, "w") as f:
        f.write("epoch,train_loss,val_loss,gpu_mb\n")

In [ ]:
# ======================================
# Training loop con validation + checkpoint 
# ======================================
from tqdm import tqdm
import torch
import torch.nn.functional as F
import os
import math

num_epochs = 20
if(start_epoch<=0):
    train_losses = []
    val_losses = []

os.makedirs(checkpoint_dir, exist_ok=True)  # crea cartella se non esiste
checkpoint_interval = 1  # salva checkpoint ogni 1 epoche

for epoch in range(start_epoch, num_epochs):
    print(f"\n🔥 Epoch {epoch}/{num_epochs} start")

    # ----------------------
    # TRAIN
    # ----------------------
    model.train()
    running_loss = 0.0
    for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc=f"Train batches Epoch {epoch}")):
        images, labels = images.to(device), labels.to(device).unsqueeze(1).float()
        optimizer.zero_grad()
        outputs = model(images)
        #controllo perchè ho problemi di NaN (risolto)
        #if torch.isnan(outputs).any() or torch.isinf(outputs).any():
        #        print("❌ NaN/Inf negli OUTPUT")
        #        print("Outputs min/max:", outputs.min().item(), outputs.max().item())
        #        break
        #controllo per problemi NaN (problema risolto)
        #print("Labels NaN:", torch.isnan(labels).any().item())
        #print("Labels Inf:", torch.isinf(labels).any().item())
        #print("Labels min/max:", labels.min().item(), labels.max().item())
        
        loss = criterion(outputs, labels)

        #if torch.isnan(loss) or torch.isinf(loss): (problema NaN risolto)
        #    print("❌ NaN nella LOSS")
        #    print("Loss value:", loss)
        #    break

        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    # ----------------------
    # VALIDATION
    # ----------------------
    model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(tqdm(val_loader, desc=f"Validation batches Epoch {epoch}")):
            images, labels = images.to(device), labels.to(device).unsqueeze(1).float()
            outputs = model(images)
            #if torch.isnan(outputs).any() or torch.isinf(outputs).any(): (problema NaN risolto)
            #   print("❌ NaN/Inf negli OUTPUT")
            #    print("Outputs min/max:", outputs.min().item(), outputs.max().item())
            #    break
            #controllo per problemi NaN (risolto)
            #print("Labels NaN:", torch.isnan(labels).any().item())
            #print("Labels Inf:", torch.isinf(labels).any().item())
            #print("Labels min/max:", labels.min().item(), labels.max().item())

            loss = criterion(outputs, labels)

            #if torch.isnan(loss) or torch.isinf(loss):
            #    print("❌ NaN nella LOSS")
            #    print("Loss value:", loss)
            #    break

            val_running_loss += loss.item() * images.size(0)

    epoch_val_loss = val_running_loss / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)
    gpu_mb = torch.cuda.memory_allocated(device) / 1024**2
    # ----------------------
    # Stampa riepilogo
    # ----------------------
    print(f"Epoch {epoch} complete | "
          f"Train Loss: {epoch_train_loss:.4f}  | "
          f"Val Loss: {epoch_val_loss:.4f}")
    print(f"GPU memory used: {gpu_mb:.1f} MB")
    #scrivo log su file 
    

    with open(log_file, "a") as f:
        f.write(
            f"{epoch},"
            f"{epoch_train_loss:.6f},"
            f"{epoch_val_loss:.6f},"
            f"{gpu_mb:.1f}\n"
        )

    # ----------------------
    # Checkpoint
    # ----------------------
    if (epoch  % checkpoint_interval == 0 or epoch == num_epochs):
        checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_losses': train_losses,
            'val_losses': val_losses
        }, checkpoint_path)
        print(f"Checkpoint salvato: {checkpoint_path}")


In [ ]:
# ======================================
# Plot andamento loss
# ======================================
plt.figure(figsize=(8,5))
plt.plot(range(0,num_epochs), train_losses, label='Train Loss')
plt.plot(range(0,num_epochs), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MAE Loss')
plt.title('Andamento Loss durante il training')
plt.legend()
plt.show()

In [ ]:
from PIL import Image
import torchvision.transforms as transforms
import torch

# Percorso immagine di test su dati reali... cattive prestazioni.
img_path = "./DatasetTazzePerRegressione/reali/10_29.jpg"
#img_path = "./DatasetTazzePerRegressione/b5/non_etichettate/b5_img_027.png"

# Trasformazione uguale a quella del training

# Carica immagine
image = Image.open(img_path).convert("RGB")
image_tensor = inference_transform(image).unsqueeze(0).to(device)  # aggiungi batch dim

# Predizione
model.eval()
with torch.no_grad():
    output = model(image_tensor)
    prediction = output.item()

print(f"Predicted ObjectsCount: {prediction:.2f}")


Il modello su dati reali commette grossi errori, sui dati sintetici avvicina molto ha un margine di errore come si nota nei valori di MSE.
Si proverà ad effettuare un fine tuning con immagini reali scattate da smartphone.